# 06 – Knowledge Transfer: Streamlit-App

**Phase:** K – Knowledge (QUA³CK)  
**Projekt:** WealthScope AI  

---

## Ziel

Dieses Notebook dokumentiert die Brücke zwischen der wissenschaftlichen Analyse  
(Notebooks 01–05) und der interaktiven Streamlit-Anwendung (`app.py` + `src/`).

## App-Architektur

Seit dem Rebuild vom 2026-07-09 ist die App modular (`src/pages/` — eine Datei pro
Seite) statt als ~5.600-Zeilen-Monolith aufgebaut. Routing läuft über Streamlits
native `st.navigation()` statt eigener Query-Parameter-Logik.

```
app.py (Entrypoint: Theme-CSS, Sidebar, st.navigation)
│
├── src/pages/  (eine Datei pro Seite)
│   ├── start.py            → 🏠 Projektübersicht, aktuelles Signal
│   ├── market.py           → 📊 Kurs-Chart, Indikatoren, Drawdown
│   ├── ml_insights.py      → 🔬 Score-Zerlegung, Korrelationsmatrix, Konfusionsmatrix,
│   │                          Lernkurve, Feature Importance, SHAP
│   ├── news_page.py / assistant_page.py → 📰 NewsAPI + Gemini-KI
│   ├── simulator.py        → 💼 Portfolio-Simulator
│   ├── kompass.py          → 🧭 Kapital-Kompass (Risiko/Positionsgröße)
│   ├── watchlist.py        → 📋 Ranking über alle 26 Ticker
│   ├── data_lab.py         → 🔬 Explorative Datenanalyse
│   └── methodology.py      → 📖 QUA³CK-Erklärung
│
├── Datenquellen
│   ├── data/processed/          → Lokaler Feature-Datensatz
│   ├── yfinance                 → Live-Daten (optional, mit Distribution-Shift-Hinweis)
│   └── NewsAPI                  → Aktuelle Finanznachrichten
│
└── ML-Modell + Diagnostik
    ├── models/wealthscope_model.joblib   → RandomForest (~55 % Accuracy, ROC-AUC ~0.59)
    ├── models/diagnostics.json           → Konfusionsmatrix, ROC/PR, Cross-Val, Feature Importance
    └── models/learning_curve.json        → Lernkurve (Bias/Variance), von
                                              scripts/train_and_diagnose.py vorberechnet
```


In [ ]:
# Verifikation der App-Struktur
from pathlib import Path
import ast

PROJECT_ROOT = Path("..").resolve()
app_path = PROJECT_ROOT / "app.py"
src_pages = PROJECT_ROOT / "src" / "pages"

if app_path.exists():
    code_text = app_path.read_text(encoding="utf-8")
    lines = code_text.splitlines()
    print(f"app.py: {len(lines):,} Zeilen (schlanker Entrypoint)")

    imports = [l.strip() for l in lines if l.startswith("import ") or l.startswith("from ")]
    print(f"Imports:    {len(imports)} Zeilen")

    # Registrierte Seiten (st.Page-Aufrufe) zählen
    page_calls = [l.strip() for l in lines if "st.Page(" in l]
    print(f"\nRegistrierte Seiten (st.Page): {len(page_calls)}")

    print("\nSeiten-Module in src/pages/:")
    if src_pages.exists():
        for f in sorted(src_pages.glob("*.py")):
            if f.name != "__init__.py":
                print(f"  - {f.name}")

    print("\nKern-Bibliotheken:")
    core_libs = ["streamlit", "pandas", "plotly", "sklearn", "yfinance", "requests", "google"]
    for lib in core_libs:
        found = any(lib in l for l in imports)
        print(f"  {'✅' if found else '❌'} {lib}")
else:
    print("app.py nicht gefunden.")


## Wie die Notebooks in die App einfließen

| Notebook | App-Seite | Transfer |
|---|---|---|
| 02 – Understanding | `data_lab.py` (Datenlabor), `methodology.py` | EDA-Ergebnisse, Missing-Value-Handling kommuniziert |
| 03 – Feature Eng. | `market.py`, `diagnostics.py` (Korrelationsmatrix) | Features berechnet & angezeigt |
| 04 – Modeling | `ml_insights.py` | RF-Modell + Konfusionsmatrix, ROC/PR, Lernkurve, Feature Importance |
| 05 – Conclude | `start.py`, Disclaimer auf jeder Seite | Grenzen transparent kommuniziert (EMH, Fama 1970) |
| 07 – NewsAPI | `news_page.py`, `assistant_page.py` | API-Integration live in App |

## Design-Entscheidungen

- **Streamlit** statt Dash/Flask: Schneller Prototyp, Python-nativ, kein JS-Overhead
- **`st.navigation`** statt eigenem Query-Parameter-Routing: seit Streamlit 1.36
  können Seiten als Funktionen (nicht nur Dateien) registriert werden — das
  ermöglicht die modulare `src/pages/`-Struktur ohne selbstgebauten Router
- **Plotly** für interaktive Charts: Hover, Zoom, Download out-of-the-box
- **Eigenes Farb-/Typografie-System** (`src/theme.py`): ein vom Fach (Finanz-Ledger,
  nicht generisches SaaS-Blau) abgeleiteter Akzent, serifenbetonte Displayschrift
  für Titel, Monospace für alle Zahlen (Kurse, Scores) — bewusst kein Indigo/
  Emoji-Standardlook
- **Gemini-KI** für kontextuellen Assistenten: Erklärt Analyseergebnisse verständlich

→ Weiter mit **07_newsapi_assistant_export.ipynb**
